In [18]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import product

import fugu
from fugu import Scaffold, Brick
from fugu.bricks import Vector_Input
from fugu.backends import snn_Backend

In [19]:
# Let's take a look at all valid input coding types.
fugu.input_coding_types

['current',
 'unary-B',
 'unary-L',
 'binary-B',
 'binary-L',
 'temporal-B',
 'temporal-L',
 'Raster',
 'Population',
 'Rate',
 'Undefined']

In [20]:
ACTIONS = ["Cooperate", "Defect"]

payoffs = {
    ("Cooperate", "Cooperate"): (1, 1),
    ("Cooperate", "Defect"): (3, 0),
    ("Defect", "Cooperate"): (0, 3),
    ("Defect", "Defect"): (2, 2),
}

def find_nash_equilibrium(actions, payoffs):
    """
    Pure strategy Nash equilibrium: outcome where no player can reduce
    their cost by unilaterally switching action.
    """
    nash = []
    for action_1, action_2 in product(actions, actions):
        cost_1, cost_2 = payoffs[(action_1, action_2)]
        player_1_can_improve = any(payoffs[(alt, action_2)][0] < cost_1 for alt in actions if alt != action_1)
        player_2_can_improve = any(payoffs[(action_1, alt)][1] < cost_2 for alt in actions if alt != action_2)
        if not player_1_can_improve and not player_2_can_improve:
            nash.append((action_1, action_2))
    return nash

true_nash = find_nash_equilibrium(ACTIONS, payoffs)
print("Ground-truth Nash equilibrium:", true_nash)

Ground-truth Nash equilibrium: [('Defect', 'Defect')]


In [27]:
class NashEquilibrium_Brick(Brick):
    def __init__(self, actions, payoffs, name=None):
        super().__init__()
        self.name = name
        self.actions = actions
        self.payoffs = payoffs
        self.is_built = False
        self.supported_codings = fugu.input_coding_types
        self.metadata = {'D': max(c for pair in payoffs.values() for c in pair) + 1}

    def build(self, graph, metadata, controlled_nodes, inputs, input_codings):
        output_codings = [input_codings[0]]

        completed_node = self.name + "_complete"
        graph.add_node(completed_node, index=-1, threshold=0.0, decay=0.0, p=1.0, potential=0.0)
        graph.add_edge(controlled_nodes[0]['complete'], completed_node, weight=1.0, delay=1.0)

        player_1_source = inputs[0][0]
        player_2_source = inputs[1][0]

        output_nodes = []
        for idx, (action_1, action_2) in enumerate(product(self.actions, self.actions)):
            cost_1, cost_2 = self.payoffs[(action_1, action_2)]
            node_name = f"{self.name}_{action_1}_{action_2}"
            graph.add_node(node_name, index=idx, threshold=1.9, decay=1.0, p=1.0, potential=0.0)
            graph.add_edge(player_1_source, node_name, weight=1.0, delay=float(cost_1 + 1))
            graph.add_edge(player_2_source, node_name, weight=1.0, delay=float(cost_2 + 1))
            output_nodes.append(node_name)

        self.is_built = True
        return (graph, self.metadata, [{'complete': completed_node}],
            [output_nodes], output_codings)

In [33]:
scaffold = Scaffold()
scaffold.add_brick(Vector_Input(np.array([1]), coding="Raster", name="P1"), 'input')
scaffold.add_brick(Vector_Input(np.array([1]), coding="Raster", name="P2"), 'input')
scaffold.add_brick(NashEquilibrium_Brick(ACTIONS, payoffs, name='Nash'), [(0, 0), (1, 0)], output=True)
scaffold.lay_bricks()
scaffold.summary(verbose=1)

backend = snn_Backend()
backend_args = {}
backend_args['record'] = 'all'
backend.compile(scaffold, backend_args)
result = backend.run(10)
print(result)

Scaffold is built: True
-------------------------------------------------------
Bricks:

Brick No.: 0
Brick Tag: P1-18
Brick Name: P1
{'tag': 'P1-18', 'name': 'P1', 'brick': <fugu.bricks.input_bricks.Vector_Input object at 0x112402090>, 'layer': 'input', 'ports': {'output': PortData(spec=PortSpec(name='output', description='', index=0, minimum=1, maximum=1, channels={'data': ChannelSpec(name='data', description='', coding='Raster', shape=(1,), required=True), 'begin': ChannelSpec(name='begin', description='', coding=[], shape=(1,), required=True), 'complete': ChannelSpec(name='complete', description='', coding=[], shape=(1,), required=True)}), channels={'data': ChannelData(spec=ChannelSpec(name='data', description='', coding='Raster', shape=(1,), required=True), neurons=['P1-18:(0,)']), 'begin': ChannelData(spec=ChannelSpec(name='begin', description='', coding=[], shape=(1,), required=True), neurons=['P1-18:begin']), 'complete': ChannelData(spec=ChannelSpec(name='complete', description

In [34]:
scaffold.graph.nodes(data=True)

result = backend.run(10)
print(result)

Empty DataFrame
Columns: [time, neuron_number]
Index: []


In [35]:
fired_pairs = []
fire_times = {}

for neuron_name, times in result.items():
    if neuron_name.startswith('Nash_') and neuron_name != 'Nash_complete':
        # name format: Nash_{a1}_{a2}
        _, action_1, action_2 = neuron_name.split('_', 2)
        if len(times) > 0:
            fired_pairs.append((action_1, action_2))
            fire_times[(action_1, action_2)] = min(times)

print("Pair neurons that fired (cost_player_1 == cost_player_2):")
for action_1, action_2 in fired_pairs:
    cost_1, cost_2 = payoffs[(action_1, action_2)]
    is_nash = (action_1, action_2) in true_nash
    print(f"  ({action_1}, {action_2})  costs = ({cost_1},{cost_2})  t = {fire_times[(action_1,action_2)]}  Nash = {is_nash}")

print()
print("Ground-truth Nash:", true_nash)
print("Spiking fired: ", fired_pairs)

Pair neurons that fired (cost_player_1 == cost_player_2):

Ground-truth Nash: [('Defect', 'Defect')]
Spiking fired:  []
